# ניתוח תביעת הוצאות

מחברת זו מדגימה כיצד ליצור סוכנים שמשתמשים בתוספים כדי לעבד הוצאות נסיעה מתמונות קבלות מקומיות, ליצור אימייל תביעת הוצאות ולהציג נתוני הוצאות באמצעות תרשים עוגה. הסוכנים בוחרים פונקציות באופן דינמי בהתאם להקשר המשימה.

שלבים:
1. סוכן OCR מעבד את תמונת הקבלה המקומית ומחלץ נתוני הוצאות נסיעה.
2. סוכן האימייל יוצר אימייל תביעת הוצאות.

### דוגמה לתרחיש הוצאות נסיעה:
דמיינו שאתם עובדים שנוסעים לפגישה עסקית בעיר אחרת. לחברה שלכם יש מדיניות להחזיר את כל ההוצאות הסבירות הקשורות לנסיעה. הנה פירוט של הוצאות נסיעה פוטנציאליות:
- תחבורה:
כרטיס טיסה הלוך ושוב מעיר המגורים שלכם לעיר היעד.
מוניות או שירותי הסעה אל ומאת נמל התעופה.
תחבורה מקומית בעיר היעד (כגון תחבורה ציבורית, השכרת רכבים או מוניות).

- לינה:
שהייה במלון במשך שלוש לילות במלון עסקי ברמה בינונית קרוב למקום הפגישה.

- ארוחות:
קצבת ארוחות יומית עבור ארוחת בוקר, צהריים וערב, בהתבסס על מדיניות הדמי היומיום של החברה.

- הוצאות שונות:
דמי חנייה בנמל התעופה.
תשלומי גישה לאינטרנט במלון.
טיפים או חיובי שירות קטנים.

- תיעוד:
אתם מגישים את כל הקבלות (טיסות, מוניות, מלון, ארוחות וכו') ודוח הוצאות מלא להחזר.


## ייבא את הספריות הנדרשות

ייבא את הספריות והמודולים הדרושים לפנקס הערות.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## הגדר מודלים להוצאות

 צור מודל פיידנטיק להוצאות בודדות ומחלקת ExpenseFormatter להמרת שאילתת משתמש לנתוני הוצאה מובנים.

 כל הוצאה תיוצג בפורמט:
 `{'date': '07-Mar-2025', 'description': 'טיסה ליעד', 'amount': 675.99, 'category': 'תחבורה'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## הגדרת כלים - יצירת האימייל

צור פונקציית כלי ליצירת אימייל להגשת תביעת הוצאה.
- כלי זה משתמש ב-`@tool` מהמסגרת Microsoft Agent.
- הוא מחשב את הסכום הכולל של ההוצאות ומעצב את הפרטים לתוך גוף האימייל.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# כלי לחילוץ הוצאות נסיעה מתמונות קבלות

צור פונקציית כלי להוצאת הוצאות נסיעה מתמונות קבלות.  
- כלי זה משתמש בקישוט `@tool` מ-Microsoft Agent Framework.  
- הוא קורא את תמונת הקבלה, מקודד אותה כ-base64, ומחזיר את URI הנתונים לניתוח על ידי הסוכן.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## עיבוד הוצאות

הגדר את הסוכנים וחבר אותם לזרימת עבודה סדרתית באמצעות `WorkflowBuilder`.
- סוכן ה-OCR מחלץ נתוני הוצאות מובנים מתמונת הקבלה באמצעות הכלי `load_receipt_image`.
- סוכן האימייל לוקח את הנתונים החילוצים ויוצר מייל תביעה להחזר הוצאות מקצועי באמצעות הכלי `generate_expense_email`.
- `WorkflowBuilder` עם `add_edge` יוצר צינור סדרתי: סוכן OCR → סוכן אימייל.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## פונקציה ראשית

בנה את תהליך העבודה הסדרתי והריץ אותו כדי לעבד את תמונת הקבלה וליצור את הודעת הדואר האלקטרוני לתביעת ההוצאה.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:  
מסמך זה תורגם באמצעות שירות תרגום מבוסס בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). אף שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עשויים להכיל שגיאות או אי-דיוקים. המסמך המקורי בשפת המקור שלו ייחשב כמקור הסמכות. לשם מידע קריטי, מומלץ להשתמש בשירותי תרגום מקצועיים של בני אדם. אנו לא נושאים באחריות לכל אי-הבנה או פרשנות שגויה הנובעים מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
